In [2]:
import json
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import time
import os
import re
import threading
from typing import Any

from dotenv import load_dotenv
from llama_index.core.llms import ChatMessage
from llama_index.core.base.llms.types import MessageRole
from llama_index.llms.openai import OpenAI

In [3]:
# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
xml_file = "Base_Model_Arch_vs_Structural_Clashes.xml"
output_json = "clashes_normalized.json"
usable_output_json = "usable_clashes.json"


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def text_or_none(node, path):
    child = node.find(path)
    return child.text.strip() if child is not None and child.text else None


def to_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def parse_name_value_nodes(parent, path):
    out = {}
    for item in parent.findall(path):
        name = text_or_none(item, "name")
        value = text_or_none(item, "value")
        if name:
            out[name] = value
    return out


def extract_created_at(clashresult):
    date_node = clashresult.find("./createddate/date")
    if date_node is None:
        return None

    try:
        return (
            f"{int(date_node.attrib.get('year')):04d}-"
            f"{int(date_node.attrib.get('month')):02d}-"
            f"{int(date_node.attrib.get('day')):02d}T"
            f"{int(date_node.attrib.get('hour', 0)):02d}:"
            f"{int(date_node.attrib.get('minute', 0)):02d}:"
            f"{int(date_node.attrib.get('second', 0)):02d}"
        )
    except Exception:
        return None


def extract_clash_point(clashresult):
    pt = clashresult.find("./clashpoint/pos3f")
    if pt is None:
        return None

    return {
        "x": to_float(pt.attrib.get("x")),
        "y": to_float(pt.attrib.get("y")),
        "z": to_float(pt.attrib.get("z")),
    }


def extract_revit_global_id(attrs):
    candidate_keys = [
        "Revit UniqueId",
        "Revit Unique ID",
        "UniqueId",
        "Unique ID",
        "GlobalId",
        "Global ID",
        "IfcGUID",
        "Ifc GUID",
    ]
    for key in candidate_keys:
        if key in attrs and attrs[key]:
            return attrs[key]
    return None

# ------------------------------------------------------------
# CORE PARSERS
# ------------------------------------------------------------
def parse_clash_object(clashobject):
    attrs = parse_name_value_nodes(clashobject, "./objectattribute")
    tags = parse_name_value_nodes(clashobject, "./smarttags/smarttag")

    base_obj = {
        "revitGlobalId": extract_revit_global_id(attrs),
        "elementId": attrs.get("Element ID"),
        "clashMetadata": {
            "layer": text_or_none(clashobject, "./layer"),
            "itemName": tags.get("Item Name"),
            "itemType": tags.get("Item Type"),
        },
        "rawAttributes": attrs,
        "rawSmartTags": tags,
    }

    return base_obj


def parse_clash_result(clashresult):
    return {
        "clashName": clashresult.attrib.get("name"),
        "clashGuid": clashresult.attrib.get("guid"),
        "clashMetadata": {
            "status": clashresult.attrib.get("status"),
            "description": text_or_none(clashresult, "./description"),
            "resultStatus": text_or_none(clashresult, "./resultstatus"),
            "distance": to_float(clashresult.attrib.get("distance")),
            "href": clashresult.attrib.get("href"),
            "gridLocation": text_or_none(clashresult, "./gridlocation"),
            "createdAt": extract_created_at(clashresult),
            "clashPoint": extract_clash_point(clashresult),
        },
        "objects": [
            parse_clash_object(obj)
            for obj in clashresult.findall("./clashobjects/clashobject")
        ],
    }


def parse_clash_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    output = {
        "sourceFile": Path(xml_path).name,
        "sourcePath": str(Path(xml_path).resolve()),
        "units": root.attrib.get("units"),
        "tests": []
    }

    for clashtest in root.findall(".//clashtest"):
        summary = clashtest.find("./summary")

        test_data = {
            "testName": clashtest.attrib.get("name"),
            "testType": clashtest.attrib.get("test_type"),
            "testStatus": clashtest.attrib.get("status"),
            "tolerance": clashtest.attrib.get("tolerance"),
            "summary": {
                "total": int(summary.attrib["total"]) if summary is not None and summary.attrib.get("total") else None,
                "new": int(summary.attrib["new"]) if summary is not None and summary.attrib.get("new") else None,
                "active": int(summary.attrib["active"]) if summary is not None and summary.attrib.get("active") else None,
                "reviewed": int(summary.attrib["reviewed"]) if summary is not None and summary.attrib.get("reviewed") else None,
                "approved": int(summary.attrib["approved"]) if summary is not None and summary.attrib.get("approved") else None,
                "resolved": int(summary.attrib["resolved"]) if summary is not None and summary.attrib.get("resolved") else None,
            },
            "clashes": []
        }

        for clashresult in clashtest.findall("./clashresults/clashresult"):
            test_data["clashes"].append(parse_clash_result(clashresult))

        output["tests"].append(test_data)
    
    return output


def trim_clash_xml(xml_path, clash_limit):
    """
    Keep only the first `clash_limit` clashresult nodes across all tests,
    then save a trimmed XML as YYYYMMDD_{original_file_name}.

    Returns:
        Path: Absolute path to the newly written XML file.
    """
    if not isinstance(clash_limit, int):
        raise TypeError("clash_limit must be an integer")
    if clash_limit < 0:
        raise ValueError("clash_limit must be >= 0")

    source_path = Path(xml_path).resolve()
    tree = ET.parse(source_path)
    root = tree.getroot()

    remaining = clash_limit

    for clashtest in root.findall(".//clashtest"):
        clashresults_node = clashtest.find("./clashresults")
        if clashresults_node is None:
            continue

        clash_nodes = list(clashresults_node.findall("./clashresult"))
        keep_count = max(0, min(len(clash_nodes), remaining))

        for idx, clash_node in enumerate(clash_nodes):
            if idx >= keep_count:
                clashresults_node.remove(clash_node)

        # Keep summary.total aligned with the trimmed clashresult count.
        summary_node = clashtest.find("./summary")
        if summary_node is not None and "total" in summary_node.attrib:
            summary_node.attrib["total"] = str(
                len(clashresults_node.findall("./clashresult"))
            )

        remaining -= keep_count

    output_name = f"{time.strftime('%Y%m%d')}_{source_path.name}"
    output_path = source_path.with_name(output_name)
    tree.write(output_path, encoding="utf-8", xml_declaration=True)

    return output_path


def optimize_clash_for_agent(clash):
    """
    Minifies a BIM clash object for LLM token optimization.
    Extracts only the ID, clash type, and core item metadata.
    """
    return {
        "id": clash.get("clashGuid"),
        "type": clash.get("clashMetadata", {}).get("description"),
        "items": [
            {
                "n": obj.get("clashMetadata", {}).get("itemName"),
                "t": obj.get("clashMetadata", {}).get("itemType")
            } 
            for obj in clash.get("objects", [])
        ]
    }

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
if __name__ == "__main__":
    # Main function to parse clash xml
    # data = parse_clash_xml(xml_file)
    # with open(output_json, "w", encoding="utf-8") as f:
    #     json.dump(data, f, indent=2, ensure_ascii=False)

    # raw_clashes = data["tests"][0]["clashes"]

    # Side quest to trim xml
    trimmed_xml = trim_clash_xml(xml_file, 100)

## Infer severity using LLM

In [3]:
PREPROMPT = """
Role: You are a BIM Coordination Engine. Your mission is to analyze multi-object BIM clashes and provide a strategic resolution plan by identifying the "Lead" (the element that stays) and the "Movers" (the elements that must reroute).

1. Trade Priority Hierarchy (High to Low):

STR (Structural): Beams, Columns, Slabs, Metal Decks.

PLUMB-G (Gravity Plumbing): Sanitary, Storm, Drainage.

MECH-L (Large Mechanical): Primary Duct Mains, AHUs.

PLUMB-P (Pressurized): Domestic Water, Gas, Fire Protection.

ELEC (Electrical): Cable Trays, Conduits, Lighting.

2. Triage Logic:

Determine Severity: * CRITICAL: Any clash containing 2 or more items from Priority 1 or 2.

MEDIUM: Any clash containing Priority 3 vs. Priority 1/2.

LOW: Clashes containing only Priority 4 and 5 items.

Identify the "Lead": The item with the highest priority in the hierarchy. If there is a tie between two items of the same priority (e.g., two structural beams), both are designated as "Lead."

3. Discipline Codes:
ARC (Architectural), STR (Structural), MECH (Mechanical), PLUMB (Plumbing), FP (Fire Protection), ELEC (Electrical).

Input Format:

JSON
{
  "id": "uuid",
  "type": "Hard | Soft",
  "items": [{ "n": "Name", "t": "Type" }, ...]
}
Output Format:
Return a JSON array of objects:

JSON
[
  {
    "clash": "uuid",
    "severity": "LOW | MEDIUM | CRITICAL",
    "disciplines": ["CODE1", "CODE2"],
    "lead": ["Item Name 1"],
  }
]
"""

In [17]:
for _env_path in (
    Path.cwd() / ".env",
    Path.cwd().parent / ".env",
    Path.cwd() / "apps" / "api" / ".env",
):
    if _env_path.is_file():
        load_dotenv(_env_path)
        break
else:
    load_dotenv()


def batch_clashes(clashes, max_batch_size=50, max_chars=None, max_workers=1):
    """
    Batches a list of optimized clash objects.

    Args:
        clashes (list): The list of optimized clash dictionaries.
        max_batch_size (int): Upper bound for clashes per batch.
        max_chars (int): Optional character limit per batch (approximate token safety).
        max_workers (int): Worker budget used for balanced count-based batching.

    Returns:
        list: A list of lists, where each sub-list is a batch.
    """
    if not clashes:
        return []

    # For token-safe mode, keep the existing greedy char-based batching behavior.
    if max_chars:
        batches = []
        current_batch = []
        current_chars = 0
        for clash in clashes:
            clash_json = json.dumps(clash)
            clash_len = len(clash_json)
            if current_batch and (current_chars + clash_len > max_chars):
                batches.append(current_batch)
                current_batch = []
                current_chars = 0
            current_batch.append(clash)
            current_chars += clash_len
        if current_batch:
            batches.append(current_batch)
        return batches

    total = len(clashes)
    max_workers = max(1, int(max_workers))
    max_batch_size = max(1, int(max_batch_size))

    min_batches_needed = (total + max_batch_size - 1) // max_batch_size
    preferred_batches = min(total, max_workers)
    batch_count = max(min_batches_needed, preferred_batches)

    base = total // batch_count
    remainder = total % batch_count

    batches = []
    start = 0
    for i in range(batch_count):
        size = base + (1 if i < remainder else 0)
        end = start + size
        batches.append(clashes[start:end])
        start = end

    return batches


def _strip_code_fence(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```\s*$", "", text)
    return text


def _clash_payload(clash: dict[str, Any], *, minify: bool) -> dict[str, Any]:
    if minify and "clashGuid" in clash:
        return optimize_clash_for_agent(clash)
    return clash


def infer_clash_severities(
    clashes: list[dict[str, Any]],
    *,
    model: str,
    api_key: str | None = None,
    preprompt: str = PREPROMPT,
    api_base: str | None = None,
    max_batch_size: int = 40,
    max_chars: int | None = None,
    minify: bool = True,
    temperature: float = 0.0,
    max_workers: int = 3,
    debug: bool = False,
) -> list[dict[str, Any]]:
    """Run batched LLM inference in parallel and preserve batch order."""
    t_global = time.perf_counter()

    def _dbg(msg: str) -> None:
        if debug:
            dt = time.perf_counter() - t_global
            print(f"[infer +{dt:8.3f}s] {msg}")

    key = api_key or os.environ.get("OPENAI_API_KEY")
    if not key:
        raise ValueError("api_key is required (or set OPENAI_API_KEY).")

    optimized = [_clash_payload(c, minify=minify) for c in clashes]
    batches = batch_clashes(
        optimized,
        max_batch_size=max_batch_size,
        max_chars=max_chars,
        max_workers=max_workers,
    )

    worker_count = 1 if len(batches) <= 1 or max_workers <= 1 else min(max_workers, len(batches))
    _dbg(
        f"clashes={len(clashes)} batches={len(batches)} max_batch_size={max_batch_size} "
        f"worker_count={worker_count} model={model} api_base={api_base or 'default'}"
    )
    _dbg("batch sizes: " + ", ".join(str(len(chunk)) for chunk in batches))

    def _infer_batch(index: int, chunk: list[dict[str, Any]]) -> tuple[int, list[dict[str, Any]]]:
        thread_name = threading.current_thread().name
        batch_start = time.perf_counter()
        _dbg(f"batch {index} START thread={thread_name} chunk_size={len(chunk)}")

        llm_init_start = time.perf_counter()
        llm = OpenAI(
            model=model,
            api_key=key,
            api_base=api_base,
            temperature=temperature,
        )
        llm_init_elapsed = time.perf_counter() - llm_init_start

        user_content = json.dumps(chunk, ensure_ascii=False)
        messages = [
            ChatMessage(role=MessageRole.SYSTEM, content=preprompt),
            ChatMessage(role=MessageRole.USER, content=user_content),
        ]

        call_start = time.perf_counter()
        resp = llm.chat(messages)
        call_elapsed = time.perf_counter() - call_start

        text = _strip_code_fence(resp.message.content or "")
        parse_start = time.perf_counter()
        parsed = json.loads(text)
        parse_elapsed = time.perf_counter() - parse_start

        if not isinstance(parsed, list):
            raise ValueError(f"Expected JSON array from model, got {type(parsed)}")

        total_elapsed = time.perf_counter() - batch_start
        _dbg(
            f"batch {index} END thread={thread_name} llm_init={llm_init_elapsed:.2f}s "
            f"chat={call_elapsed:.2f}s parse={parse_elapsed:.3f}s total={total_elapsed:.2f}s"
        )
        return index, parsed

    if worker_count == 1:
        ordered_results = [_infer_batch(index, chunk) for index, chunk in enumerate(batches)]
    else:
        ordered_results: list[tuple[int, list[dict[str, Any]]]] = []
        with ThreadPoolExecutor(max_workers=worker_count) as executor:
            futures = [
                executor.submit(_infer_batch, index, chunk)
                for index, chunk in enumerate(batches)
            ]
            for future in as_completed(futures):
                ordered_results.append(future.result())

    out: list[dict[str, Any]] = []
    for _, parsed in sorted(ordered_results, key=lambda item: item[0]):
        out.extend(parsed)

    _dbg(f"done total_results={len(out)} total_time={time.perf_counter() - t_global:.2f}s")
    return out

In [18]:

test_clashes = raw_clashes[:60]
t0 = time.perf_counter()
max_batch_size = 40
max_workers = 6
severities = infer_clash_severities(
    test_clashes,
    # model=os.environ["MODEL_NAME"],
    # api_base=os.environ.get("OPENAI_BASE_URL") or None,
    model="gpt-4o-mini",
    max_batch_size=max_batch_size,
    max_workers=max_workers,
    debug=True,
)
elapsed = time.perf_counter() - t0
print(
    f"infer_clash_severities: {elapsed:.2f}s wall time "
    f"max_batch_size={max_batch_size}, max_workers={max_workers}, "
    f"clashes={len(test_clashes)}, results={len(severities)}"
)
print(severities)  # optional: comment this out while debugging timing

[infer +   0.000s] clashes=60 batches=6 max_batch_size=40 worker_count=6 model=gpt-4o-mini api_base=default
[infer +   0.000s] batch sizes: 10, 10, 10, 10, 10, 10
[infer +   0.001s] batch 0 START thread=ThreadPoolExecutor-3_0 chunk_size=10
[infer +   0.002s] batch 1 START thread=ThreadPoolExecutor-3_1 chunk_size=10
[infer +   0.003s] batch 2 START thread=ThreadPoolExecutor-3_2 chunk_size=10
[infer +   0.004s] batch 3 START thread=ThreadPoolExecutor-3_3 chunk_size=10
[infer +   0.006s] batch 4 START thread=ThreadPoolExecutor-3_4 chunk_size=10
[infer +   0.007s] batch 5 START thread=ThreadPoolExecutor-3_5 chunk_size=10
[infer +  10.958s] batch 5 END thread=ThreadPoolExecutor-3_5 llm_init=0.00s chat=10.95s parse=0.000s total=10.95s
[infer +  11.193s] batch 2 END thread=ThreadPoolExecutor-3_2 llm_init=0.00s chat=11.19s parse=0.000s total=11.19s
[infer +  11.718s] batch 3 END thread=ThreadPoolExecutor-3_3 llm_init=0.00s chat=11.71s parse=0.000s total=11.71s
[infer +  12.419s] batch 1 END th

In [6]:
# Example (after load_dotenv cell): MODEL_NAME, OPENAI_API_KEY, OPENAI_BASE_URL from .env
# severities = infer_clash_severities(
#     usable_clashes[:120],
#     model=os.environ["MODEL_NAME"],
#     api_base=os.environ.get("OPENAI_BASE_URL") or None,
#     max_batch_size=40,
# )
